# 第8章: ニューラルネット

第7章で取り組んだポジネガ分類を題材として、ニューラルネットワークで分類モデルを実装する。なお、この章ではPyTorchやTensorFlow、JAXなどの深層学習フレームワークを活用せよ。

## 70. 単語埋め込みの読み込み

事前学習済み単語埋め込みを活用し、$|V| \times d_\rm{emb}$ の単語埋め込み行列$\pmb{E}$を作成せよ。ここで、$|V|$は単語埋め込みの語彙数、$d_\rm{emb}$は単語埋め込みの次元数である。ただし、単語埋め込み行列の先頭の行ベクトル$\pmb{E}_{0,:}$は、将来的にパディング（`<PAD>`）トークンの埋め込みベクトルとして用いたいので、ゼロベクトルとして予約せよ。ゆえに、$\pmb{E}$の2行目以降に事前学習済み単語埋め込みを読み込むことになる。

もし、Google Newsデータセットの[学習済み単語ベクトル](https://drive.google.com/file/d/0B7XkCwpI5KDYNlNUTTlSS21pQmM/edit?usp=sharing)（300万単語・フレーズ、300次元）を全て読み込んだ場合、$|V|=3000001, d_\rm{emb}=300$になるはずである（ただ、300万単語の中には、殆ど用いられない稀な単語も含まれるので、語彙を削減した方がメモリの節約になる）。

また、単語埋め込み行列の構築と同時に、単語埋め込み行列の各行のインデックス番号（トークンID）と、単語（トークン）への双方向の対応付けを保持せよ。

In [3]:
!pip install gensim
import gensim.downloader as api
import numpy as np

print("Google News Word2Vecモデルをダウンロード中...")
model = api.load("word2vec-google-news-300")
print("モデルのダウンロードと読み込みが完了")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 73.8 MB/s eta 0:00:00
Google News Word2Vecモデルをダウンロード中...
[==================================================] 100.0% 1662.8/1662.8MB downloaded
モデルのダウンロードと読み込みが完了


In [4]:
VOCAB_SIZE = len(model.key_to_index) + 1  # 300万単語 + <PAD>分
EMB_DIM = model.vector_size           # 300次元

#単語埋め込み行列
E = np.zeros((VOCAB_SIZE, EMB_DIM), dtype=np.float32)

word2id = {"<PAD>": 0}
id2word = {0: "<PAD>"}

#単語ベクトルを行列に格納し、辞書を構築
print("行列の構築中...")
for i, word in enumerate(model.index_to_key, start=1):
    E[i] = model[word]      # 行列のi行目にベクトルを格納
    word2id[word] = i      # 単語 -> ID
    id2word[i] = word      # ID -> 単語

#確認
print(f"行列の形状: {E.shape}")

行列の構築中...
行列の形状: (3000001, 300)
ID 0 (<PAD>) のベクトル (最初の5要素): [0. 0. 0. 0. 0.]
ID 1 (</s>) のベクトル (最初の5要素): [ 0.00112915 -0.00089645  0.00031853  0.00153351  0.00110626]


## 71. データセットの読み込み

[General Language Understanding Evaluation (GLUE)](https://gluebenchmark.com/) ベンチマークで配布されている[Stanford Sentiment Treebank (SST)](https://dl.fbaipublicfiles.com/glue/data/SST-2.zip) をダウンロードし、訓練セット（train.tsv）と開発セット（dev.tsv）のテキストと極性ラベルと読み込み、全てのテキストをトークンID列に変換せよ。このとき、単語埋め込みの語彙でカバーされていない単語は無視し、トークン列に含めないことにせよ。また、テキストの全トークンが単語埋め込みの語彙に含まれておらず、空のトークン列となってしまう事例は、訓練セットおよび開発セットから削除せよ（このため、第7章の実験で得られた正解率と比較できなくなることに注意せよ）。

事例の表現方法は任意でよいが、例えば"contains no wit , only labored gags"がネガティブに分類される事例は、次のような辞書オブジェクトで表現すればよい。

```
{'text': 'contains no wit , only labored gags',
 'label': tensor([0.]),
 'input_ids': tensor([ 3475,    87, 15888,    90, 27695, 42637])}
```

この例では、`text`はテキスト、`label`は分類ラベル（ポジティブなら`tensor([1.])`、ネガティブなら`tensor([0.])`）、`input_ids`はテキストのトークン列をID列で表現している。

In [5]:
!wget https://dl.fbaipublicfiles.com/glue/data/SST-2.zip
!unzip SST-2.zip

import torch
import pandas as pd

def load_sst2_and_idize(filepath, word2id):
    df = pd.read_csv(filepath, sep='\t')
    dataset = []

    for text, label in zip(df['sentence'], df['label']):
        # 簡易的なトークナイズ（半角スペースで分割）
        tokens = text.split()
        # 語彙に含まれる単語のみをIDに変換
        input_ids = [word2id[word] for word in tokens if word in word2id]

        # 空のトークン列は除外
        if len(input_ids) > 0:
            dataset.append({
                'text': text,
                'label': torch.tensor([float(label)]),
                'input_ids': torch.tensor(input_ids)
            })
    return dataset

# データの読み込みと変換
train_data = load_sst2_and_idize('SST-2/train.tsv', word2id)
dev_data = load_sst2_and_idize('SST-2/dev.tsv', word2id)

# 確認
print(f"訓練データ数: {len(train_data)}")
print(f"開発データ数: {len(dev_data)}")
print("サンプル:", train_data[0])

--2026-05-29 03:06:52--  https://dl.fbaipublicfiles.com/glue/data/SST-2.zip
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 18.238.192.99, 18.238.192.12, 18.238.192.60, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|18.238.192.99|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7439277 (7.1M) [application/zip]
Saving to: ‘SST-2.zip’

SST-2.zip           100%[===================>]   7.09M  --.-KB/s    in 0.1s    

2026-05-29 03:06:52 (49.4 MB/s) - ‘SST-2.zip’ saved [7439277/7439277]

Archive:  SST-2.zip
   creating: SST-2/
  inflating: SST-2/dev.tsv           
   creating: SST-2/original/
  inflating: SST-2/original/README.txt  
  inflating: SST-2/original/SOStr.txt  
  inflating: SST-2/original/STree.txt  
  inflating: SST-2/original/datasetSentences.txt  
  inflating: SST-2/original/datasetSplit.txt  
  inflating: SST-2/original/dictionary.txt  
  inflating: SST-2/original/original_rt_snippets.txt  
  inflating: SST-2/original/s

## 72. Bag of wordsモデルの構築

単語埋め込みの平均ベクトルでテキストの特徴ベクトルを表現し、重みベクトルとの内積でポジティブ及びネガティブを分類するニューラルネットワーク（ロジスティック回帰モデル）を設計せよ。

In [6]:
import torch.nn as nn

class BoWModel(nn.Module):
    def __init__(self, weights_matrix, embed_dim):
        super().__init__()
        # 1. 事前学習済み埋め込みのロード（勾配更新なし）
        self.embedding = nn.Embedding.from_pretrained(torch.from_numpy(weights_matrix), freeze=True)
        # 2. 線形層 (300次元 -> 1次元)
        self.fc = nn.Linear(embed_dim, 1)

    def forward(self, input_ids):
        # input_ids: [sequence_length]
        # 埋め込み取得: [sequence_length, embed_dim]
        embeds = self.embedding(input_ids)
        # 平均ベクトル算出: [embed_dim]
        feature_vec = torch.mean(embeds, dim=0)
        # 線形層 + シグモイド
        logits = self.fc(feature_vec)
        return torch.sigmoid(logits)

# モデルのインスタンス化
model_bow = BoWModel(E, EMB_DIM)
print(model_bow)

BoWModel(
  (embedding): Embedding(3000001, 300)
  (fc): Linear(in_features=300, out_features=1, bias=True)
)


## 73. モデルの学習

問題72で設計したモデルの重みベクトルを訓練セット上で学習せよ。ただし、学習中は単語埋め込み行列の値を固定せよ（単語埋め込み行列のファインチューニングは行わない）。また、学習時に損失値を表示するなど、学習の進捗状況をモニタリングできるようにせよ。

In [7]:
from torch.utils.data import DataLoader
import torch.optim as optim

# ハイパーパラメータの設定
LEARNING_RATE = 0.1
EPOCHS = 10

# 損失関数とオプティマイザの定義
criterion = nn.BCELoss()
optimizer = optim.SGD(model_bow.parameters(), lr=LEARNING_RATE)

# 学習ループ
print("学習を開始します...")
for epoch in range(EPOCHS):
    model_bow.train()
    total_loss = 0

    # ミニバッチ化せずに1事例ずつ学習（問題73の段階的な実装として）
    for item in train_data:
        input_ids = item['input_ids']
        label = item['label']

        # 勾配の初期化
        optimizer.zero_grad()

        # 順伝播
        output = model_bow(input_ids)

        # 損失の計算
        loss = criterion(output, label)

        # 逆伝播 + 重み更新
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_data)
    print(f"Epoch [{epoch+1}/{EPOCHS}], Loss: {avg_loss:.4f}")

print("学習完了")

学習を開始します...
Epoch [1/10], Loss: 0.3884
Epoch [2/10], Loss: 0.3767
Epoch [3/10], Loss: 0.3763
Epoch [4/10], Loss: 0.3762
Epoch [5/10], Loss: 0.3762
Epoch [6/10], Loss: 0.3762
Epoch [7/10], Loss: 0.3762
Epoch [8/10], Loss: 0.3762
Epoch [9/10], Loss: 0.3762
Epoch [10/10], Loss: 0.3762
学習完了


## 74. モデルの評価

問題73で学習したモデルの開発セットにおける正解率を求めよ。

In [8]:
def calculate_accuracy(model, dataset):
    model.eval()
    correct = 0
    with torch.no_grad():
        for item in dataset:
            input_ids = item['input_ids']
            label = item['label']

            # 予測値の計算
            output = model(input_ids)
            prediction = 1 if output >= 0.5 else 0

            if prediction == label.item():
                correct += 1

    return correct / len(dataset)

# 開発セットでの正解率を表示
accuracy = calculate_accuracy(model_bow, dev_data)
print(f"開発セットの正解率: {accuracy:.4f}")

開発セットの正解率: 0.7867


## 75. パディング

複数の事例が与えられたとき、これらをまとめて一つのテンソル・オブジェクトで表現する関数`collate`を実装せよ。与えられた複数の事例のトークン列の長さが異なるときは、トークン列の長さが最も長いものに揃え、0番のトークンIDでパディングをせよ。さらに、トークン列の長さが長いものから順に、事例を並び替えよ。

例えば、訓練データセットの冒頭の4事例が次のように表されているとき、

```
[{'text': 'hide new secretions from the parental units',
  'label': tensor([0.]),
  'input_ids': tensor([  5785,     66, 113845,     18,     12,  15095,   1594])},
 {'text': 'contains no wit , only labored gags',
  'label': tensor([0.]),
  'input_ids': tensor([ 3475,    87, 15888,    90, 27695, 42637])},
 {'text': 'that loves its characters and communicates something rather beautiful about human nature',
  'label': tensor([1.]),
  'input_ids': tensor([    4,  5053,    45,  3305, 31647,   348,   904,  2815,    47,  1276,  1964])},
 {'text': 'remains utterly satisfied to remain the same throughout',
  'label': tensor([0.]),
  'input_ids': tensor([  987, 14528,  4941,   873,    12,   208,   898])}]
```

`collate`関数を通した結果は以下のようになることが想定される。

```
{'input_ids': tensor([
    [     4,   5053,     45,   3305,  31647,    348,    904,   2815,     47,   1276,   1964],
    [  5785,     66, 113845,     18,     12,  15095,   1594,      0,      0,      0,      0],
    [   987,  14528,   4941,    873,     12,    208,    898,      0,      0,      0,      0],
    [  3475,     87,  15888,     90,  27695,  42637,      0,      0,      0,      0,      0]]),
 'label': tensor([
    [1.],
    [0.],
    [0.],
    [0.]])}
```


## 77. GPU上での学習

問題76のモデル学習をGPU上で実行せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

## 78. 単語埋め込みのファインチューニング

問題77の学習において、単語埋め込みのパラメータも同時に更新するファインチューニングを導入せよ。また、学習したモデルの開発セットにおける正解率を求めよ。

## 79. アーキテクチャの変更

ニューラルネットワークのアーキテクチャを自由に変更し、モデルを学習せよ。また、学習したモデルの開発セットにおける正解率を求めよ。例えば、テキストの特徴ベクトル（単語埋め込みの平均ベクトル）に対して多層のニューラルネットワークを通したり、畳み込みニューラルネットワーク（CNN; Convolutional Neural Network）や再帰型ニューラルネットワーク（RNN; Recurrent Neural Network）などのモデルの学習に挑戦するとよい。